# 03 — CICIDS2017 Temporal Split

This notebook documents the temporal train / validation / test split applied in `preprocess.py` and explains **why** it matters for an intrusion detection system.

**Goals:**
- Visualise attack patterns across the 5-day capture period
- Validate that split boundaries are clean (no leakage between files)
- Show label distributions per split and per source file
- Inspect the feature matrices used by the models (X_train, X_valid, X_test)

**Prerequisites:** `ingest.py` and `preprocess.py` must have run successfully.

## Why temporal splitting?

Random splitting on time-series network data causes **data leakage**: the model sees future traffic patterns during training. A temporal split mimics real deployment — the model trains on past data and is tested on future data it has never seen.

| Split | Source files | Days | Attack types |
|---|---|---|---|
| **Train** | Mon, Tue, Wed | July 3–5 | BENIGN, FTP-Patator, SSH-Patator, DoS variants |
| **Validation** | Thu morning/afternoon | July 6 | Web Attacks, Infiltration |
| **Test** | Fri morning/afternoon | July 7 | PortScan, DDoS |

Critically: **validation and test contain attack types not seen in training** — this makes evaluation realistic.

## 0. Setup

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

PROCESSED = "../data/processed"

PATHS = {
    "train"   : os.path.join(PROCESSED, "cicids_train.parquet"),
    "valid"   : os.path.join(PROCESSED, "cicids_valid.parquet"),
    "test"    : os.path.join(PROCESSED, "cicids_test.parquet"),
    "X_train" : os.path.join(PROCESSED, "X_train.parquet"),
    "X_valid" : os.path.join(PROCESSED, "X_valid.parquet"),
    "X_test"  : os.path.join(PROCESSED, "X_test.parquet"),
    "y_train" : os.path.join(PROCESSED, "y_train.parquet"),
    "y_valid" : os.path.join(PROCESSED, "y_valid.parquet"),
    "y_test"  : os.path.join(PROCESSED, "y_test.parquet"),
}

missing = [k for k, v in PATHS.items() if not os.path.exists(v)]
if missing:
    print(f"Missing files: {missing}")
    print("Run: python -m src.run_pipeline --only preprocess")
else:
    print("All required files found.")

## 1. Load the splits

In [ ]:
train = pd.read_parquet(PATHS["train"])
valid = pd.read_parquet(PATHS["valid"])
test  = pd.read_parquet(PATHS["test"])

X_train = pd.read_parquet(PATHS["X_train"])
X_valid = pd.read_parquet(PATHS["X_valid"])
X_test  = pd.read_parquet(PATHS["X_test"])

y_train = pd.read_parquet(PATHS["y_train"])["Label"]
y_valid = pd.read_parquet(PATHS["y_valid"])["Label"]
y_test  = pd.read_parquet(PATHS["y_test"])["Label"]

print(f"X_train : {X_train.shape}  y_train : {y_train.shape}")
print(f"X_valid : {X_valid.shape}  y_valid : {y_valid.shape}")
print(f"X_test  : {X_test.shape}  y_test  : {y_test.shape}")

## 2. Source file → split mapping

Each source file maps to exactly one split — no file spans multiple splits.

In [ ]:
for name, df in [("TRAIN", train), ("VALIDATION", valid), ("TEST", test)]:
    files = sorted(df["source_file"].unique())
    print(f"\n{name}:")
    for f in files:
        n = len(df[df["source_file"] == f])
        print(f"  {f:<65} {n:>8,} rows")

## 3. Label distribution per split

In [ ]:
all_labels = sorted(set(train["Label"]) | set(valid["Label"]) | set(test["Label"]))

rows = []
for label in all_labels:
    rows.append({
        "Label"      : label,
        "Train"      : (train["Label"] == label).sum(),
        "Validation" : (valid["Label"] == label).sum(),
        "Test"       : (test["Label"]  == label).sum(),
    })

split_df = pd.DataFrame(rows).set_index("Label")
split_df["Total"] = split_df.sum(axis=1)
split_df = split_df.sort_values("Total", ascending=False)

print("Label counts per split (0 = label not present in that split):")
print(split_df.to_string())

In [ ]:
attack_only = split_df.drop("BENIGN").drop("Total", axis=1)

fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(attack_only))
width = 0.28

ax.bar(x - width, attack_only["Train"],      width, label="Train",      color="#1565C0")
ax.bar(x,         attack_only["Validation"], width, label="Validation", color="#EF6C00")
ax.bar(x + width, attack_only["Test"],       width, label="Test",       color="#2E7D32")

ax.set_xticks(x)
ax.set_xticklabels(attack_only.index, rotation=35, ha="right", fontsize=9)
ax.set_ylabel("Flow count")
ax.set_title("Attack class distribution per split")
ax.legend()
ax.set_yscale("symlog")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()

print("\nNote: symlog scale — some classes only appear in one split (realistic: new attack types at test time)")

## 4. Leakage check — confirm no file appears in multiple splits

In [ ]:
train_files = set(train["source_file"].unique())
valid_files = set(valid["source_file"].unique())
test_files  = set(test["source_file"].unique())

tv_overlap = train_files & valid_files
tt_overlap = train_files & test_files
vt_overlap = valid_files & test_files

print(f"Train ∩ Validation overlap : {tv_overlap if tv_overlap else 'None ✓'}")
print(f"Train ∩ Test overlap       : {tt_overlap if tt_overlap else 'None ✓'}")
print(f"Validation ∩ Test overlap  : {vt_overlap if vt_overlap else 'None ✓'}")

if not any([tv_overlap, tt_overlap, vt_overlap]):
    print("\n✓ No data leakage — splits are clean.")

## 5. Feature matrix inspection

The X_* files are the model-ready feature matrices — metadata columns (Label, dataset, source_file) are removed.

In [ ]:
print(f"Features in X_train: {X_train.shape[1]}")
print(f"Feature names: {list(X_train.columns[:10])} ...")

# Check that train/valid/test share the same columns
same_cols = list(X_train.columns) == list(X_valid.columns) == list(X_test.columns)
print(f"\nAll splits share the same feature columns: {same_cols} {'✓' if same_cols else '✗'}")

In [ ]:
# Null / inf check on modelling features
for name, X in [("X_train", X_train), ("X_valid", X_valid), ("X_test", X_test)]:
    nulls = X.isnull().sum().sum()
    infs  = np.isinf(X.select_dtypes(include=[np.number])).sum().sum()
    print(f"{name}: nulls={nulls:,}  infs={infs:,}  {'✓ clean' if nulls==0 and infs==0 else '✗ needs fixing'}")

## 6. Binary labels — how the pipeline converts multiclass to binary

`train_model.py` converts the multiclass label to binary: `BENIGN → 0`, everything else → `1`.

In [ ]:
def to_binary(y):
    return y.apply(lambda x: 0 if x == "BENIGN" else 1)

for name, y in [("Train", y_train), ("Validation", y_valid), ("Test", y_test)]:
    y_bin = to_binary(y)
    n0 = (y_bin == 0).sum()
    n1 = (y_bin == 1).sum()
    print(f"{name:<12}: BENIGN={n0:>8,} ({n0/len(y_bin)*100:.1f}%)  ATTACK={n1:>7,} ({n1/len(y_bin)*100:.1f}%)")

## Summary

| Property | Train | Validation | Test |
|---|---|---|---|
| Days | Mon–Wed | Thu | Fri |
| Attack types | FTP/SSH brute-force, DoS variants | Web attacks, Infiltration | PortScan, DDoS |
| Data leakage | None ✓ | None ✓ | None ✓ |
| Nulls/inf | 0 ✓ | 0 ✓ | 0 ✓ |

Key insight: **Validation and test contain unseen attack types** — this is the hardest and most realistic evaluation setting. Models that generalise well here are genuinely learning attack signatures rather than memorising specific flows.

**Next:** `04_model_training.ipynb` (or run `python -m src.run_pipeline --from train_binary_rf`)